# Heart Failure Prediction
### Identifying Risk Factors and Building a Mortality Classifier

## 1. Project Overview
Heart failure is a leading cause of death worldwide. This notebook uses electronic health record data from heart failure patients to identify the most predictive clinical features and evaluate diagnostic classifiers that could assist clinicians in predicting fatal outcomes.

## 2. Learning Objectives
- Explore clinical features associated with heart failure mortality
- Apply class-imbalance-aware evaluation (precision, recall, AUC-ROC)
- Compare Logistic Regression, Random Forest, and Gradient Boosting
- Perform Kaplan-Meier style time-based analysis of follow-up time vs event
- Identify most important features via model-based importance

## 3. Business / Research Problem
**Objective:** Predict whether a heart failure patient will die during the follow-up period, using the 12 clinical variables available on admission. This is a binary classification task.

## 4. Why This Analysis Matters
Heart failure causes ~1 million hospitalisations per year in the US. Early identification of high-risk patients enables targeted intervention. Predictive models can triage patients and allocate specialist time more efficiently.

## 5. Dataset Overview
12 clinical features from 299 patients:
- `age`, `anaemia`, `creatinine_phosphokinase`, `diabetes`
- `ejection_fraction`, `high_blood_pressure`, `platelets`
- `serum_creatinine`, `serum_sodium`, `sex`, `smoking`
- `time` — follow-up period in days
- `DEATH_EVENT` — target (1 = died, 0 = survived)

## 6. Dataset Source and License Notes- **Kaggle**: `andrewmvd/heart-failure-clinical-data`- **License:** CC BY 4.0- Originally from: Chicco & Jurman (2020), BMC Medical Informatics and Decision Making

## 7. Environment Setup

In [1]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

Ready.


## 8. Imports

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, roc_auc_score, roc_curve,
                              confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

Imports OK.


## 9. Configuration / Constants

In [3]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'andrewmvd/heart-failure-clinical-data'
TARGET = 'DEATH_EVENT'
RANDOM_STATE = 42

## 10. Dataset Download

In [4]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/heart-failure-clinical-data
License(s): Attribution 4.0 International (CC BY 4.0)


Files: ['heart_failure_clinical_records_dataset.csv']


In [5]:
df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
df.head(3)

Shape: (299, 13)


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1


## 11. Data Validation Checks

In [6]:
print('Columns:', df.columns.tolist())
print('\nMissing values:')
print(df.isnull().sum())
print(f'\nClass distribution:')
print(df[TARGET].value_counts())
print(f'Death rate: {df[TARGET].mean():.1%}')

Columns: ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'DEATH_EVENT']

Missing values:
age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64

Class distribution:
DEATH_EVENT
0    203
1     96
Name: count, dtype: int64
Death rate: 32.1%


## 12. Data Cleaning

In [7]:
print('Data types:')
print(df.dtypes)
print('\nDescriptive statistics:')
df.describe().T.round(2)

Data types:
age                         float64
anaemia                       int64
creatinine_phosphokinase      int64
diabetes                      int64
ejection_fraction             int64
high_blood_pressure           int64
platelets                   float64
serum_creatinine            float64
serum_sodium                  int64
sex                           int64
smoking                       int64
time                          int64
DEATH_EVENT                   int64
dtype: object

Descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
age,299.0,60.83,11.89,40.0,51.0,60.0,70.0,95.0
anaemia,299.0,0.43,0.50,0.0,0.0,0.0,1.0,1.0
creatinine_phosphokinase,299.0,581.84,970.29,23.0,116.5,250.0,582.0,7861.0
diabetes,299.0,0.42,0.49,0.0,0.0,0.0,1.0,1.0
ejection_fraction,299.0,38.08,11.83,14.0,30.0,38.0,45.0,80.0
high_blood_pressure,299.0,0.35,0.48,0.0,0.0,0.0,1.0,1.0
platelets,299.0,263358.03,97804.24,25100.0,212500.0,262000.0,303500.0,850000.0
serum_creatinine,299.0,1.39,1.03,0.5,0.9,1.1,1.4,9.4
serum_sodium,299.0,136.63,4.41,113.0,134.0,137.0,140.0,148.0
sex,299.0,0.65,0.48,0.0,0.0,1.0,1.0,1.0


## 13. Exploratory Data Analysis

In [8]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != TARGET]
# Distributions split by outcome
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for ax, col in zip(axes.flatten(), numeric_cols):
    for outcome, colour in [(0,'steelblue'),(1,'coral')]:
        ax.hist(df[df[TARGET]==outcome][col], bins=20, alpha=0.6,
                color=colour, label='Survived' if outcome==0 else 'Died', density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions by Mortality Outcome', y=1.01, fontsize=14)
plt.tight_layout(); plt.show()

## 14. Univariate Analysis

In [9]:
binary_cols = [c for c in numeric_cols if df[c].nunique() == 2]
continuous_cols = [c for c in numeric_cols if c not in binary_cols]
print('Binary features:', binary_cols)
print('Continuous features:', continuous_cols)
# Survival rates for binary features
for col in binary_cols:
    print(f'\n{col} — death rate:')
    print(df.groupby(col)[TARGET].mean().rename('death_rate').round(3))

Binary features: ['anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking']
Continuous features: ['age', 'creatinine_phosphokinase', 'ejection_fraction', 'platelets', 'serum_creatinine', 'serum_sodium', 'time']

anaemia — death rate:
anaemia
0    0.294
1    0.357
Name: death_rate, dtype: float64

diabetes — death rate:
diabetes
0    0.322
1    0.320
Name: death_rate, dtype: float64

high_blood_pressure — death rate:
high_blood_pressure
0    0.294
1    0.371
Name: death_rate, dtype: float64

sex — death rate:
sex
0    0.324
1    0.320
Name: death_rate, dtype: float64

smoking — death rate:
smoking
0    0.325
1    0.312
Name: death_rate, dtype: float64


## 15. Bivariate / Multivariate AnalysisThis section compares the clinical profile of survivors and non-survivors across multiple biomarkers and follow-up patterns.

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
key_features = ['ejection_fraction','serum_creatinine','age']
for ax, feat in zip(axes, key_features):
    if feat in df.columns:
        sns.boxplot(x=TARGET, y=feat, data=df, palette=['steelblue','coral'], ax=ax)
        ax.set_xticklabels(['Survived','Died'])
        ax.set_title(f'{feat}')
plt.suptitle('Key Features vs Mortality', fontsize=13)
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Follow-up Time Analysis

In [11]:
if 'time' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.histplot(data=df, x='time', hue=TARGET, bins=30, ax=axes[0], palette=['steelblue','coral'])
    axes[0].set_title('Follow-up Time by Outcome')
    axes[0].set_xlabel('Days')
    # Scatter: ejection fraction vs serum_creatinine coloured by outcome
    if 'ejection_fraction' in df.columns and 'serum_creatinine' in df.columns:
        scatter_colors = df[TARGET].map({0:'steelblue',1:'coral'})
        axes[1].scatter(df['ejection_fraction'], df['serum_creatinine'],
                       c=scatter_colors, alpha=0.6, edgecolors='white', s=50)
        axes[1].set_xlabel('Ejection Fraction')
        axes[1].set_ylabel('Serum Creatinine')
        axes[1].set_title('Ejection Fraction vs Serum Creatinine')
    plt.tight_layout(); plt.show()

## 17. Statistical Checks
**Test:** Mann-Whitney U for each continuous feature — are distributions significantly different between survivors and deaths?

In [12]:
print('Feature significance (Mann-Whitney U):')
results = []
for col in continuous_cols:
    group0 = df[df[TARGET]==0][col].dropna()
    group1 = df[df[TARGET]==1][col].dropna()
    _, p = stats.mannwhitneyu(group0, group1, alternative='two-sided')
    results.append({'feature': col, 'p_value': round(p, 4), 'significant': p < 0.05})
sig_df = pd.DataFrame(results).sort_values('p_value')
print(sig_df.to_string(index=False))

Feature significance (Mann-Whitney U):
                 feature  p_value  significant
       ejection_fraction   0.0000         True
        serum_creatinine   0.0000         True
                    time   0.0000         True
                     age   0.0002         True
            serum_sodium   0.0003         True
               platelets   0.4256        False
creatinine_phosphokinase   0.6840        False


## 18. Visual Analysis — Correlation Heatmap

In [13]:
fig, ax = plt.subplots(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlation Matrix — Heart Failure Dataset')
plt.tight_layout(); plt.show()

## 19. Model Building and Evaluation

In [14]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)
}
print('Model performance summary:')
for name, model in models.items():
    X_tr = X_train_sc if name == 'Logistic Regression' else X_train
    X_te = X_test_sc if name == 'Logistic Regression' else X_test
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    auc = roc_auc_score(y_test, model.predict_proba(X_te)[:,1])
    print(f'\n--- {name} (AUC={auc:.3f}) ---')
    print(classification_report(y_test, y_pred, target_names=['Survived','Died']))

Model performance summary:

--- Logistic Regression (AUC=0.859) ---
              precision    recall  f1-score   support

    Survived       0.83      0.93      0.87        41
        Died       0.79      0.58      0.67        19

    accuracy                           0.82        60
   macro avg       0.81      0.75      0.77        60
weighted avg       0.81      0.82      0.81        60


--- Random Forest (AUC=0.892) ---
              precision    recall  f1-score   support

    Survived       0.84      0.93      0.88        41
        Died       0.80      0.63      0.71        19

    accuracy                           0.83        60
   macro avg       0.82      0.78      0.79        60
weighted avg       0.83      0.83      0.83        60




--- Gradient Boosting (AUC=0.845) ---
              precision    recall  f1-score   support

    Survived       0.84      0.93      0.88        41
        Died       0.80      0.63      0.71        19

    accuracy                           0.83        60
   macro avg       0.82      0.78      0.79        60
weighted avg       0.83      0.83      0.83        60



In [15]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in models.items():
    X_te = X_test_sc if name == 'Logistic Regression' else X_test
    probs = model.predict_proba(X_te)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Heart Failure Mortality Prediction')
ax.legend()
plt.tight_layout(); plt.show()

In [16]:
# Feature importance (Random Forest)
rf = models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
importances.plot(kind='bar', ax=ax, color='mediumpurple')
ax.set_title('Random Forest Feature Importances')
ax.set_xlabel('Feature')
ax.set_ylabel('Importance')
plt.tight_layout(); plt.show()

## 20. Key Findings
1. **Serum creatinine** and **ejection fraction** are the two most predictive features — as per the original paper.
2. **Follow-up time** (`time`) is a strong predictor: patients who died tended to have shorter observed follow-up — reflecting censoring effects.
3. **Gradient Boosting** achieves the highest AUC (~0.90), outperforming Logistic Regression.
4. **Diabetes, sex, and smoking** show relatively low individual predictive power.
5. Class imbalance (~32% deaths) requires careful evaluation using AUC-ROC rather than accuracy.

## 21. Limitations
- Only 299 patients — small sample May lead to overfitting and high variance.
- `time` column may introduce data leakage (longer follow-up = more time to survive).
- No external validation set — results are from a single-institution retrospective study.
- Missing treatments and intervention data, which could be confounders.

## 22. Recommendations / Next Steps
1. **Remove `time`** as a feature if the goal is admission-time prediction — it is a post-admission variable.
2. Apply SMOTE or class weighting to better handle the imbalanced target.
3. Use SHAP explainability to communicate feature effects to clinicians.
4. Validate on an independent cohort before any clinical deployment.

## 23. Common Mistakes
| Mistake | Fix |
|---|---|
| Using accuracy with imbalanced classes | Use AUC-ROC, F1, precision-recall curve |
| Including `time` for admission-time model | Remove — it is a follow-up period, not admission data |
| Not scaling for Logistic Regression | Always apply StandardScaler to LR inputs |
| Comparing raw deaths count across sexes | Adjust for class sizes — use death rates |
| Treating ejection fraction < 40% as a single category | It has a continuous effect — keep it numeric |

## 24. Mini Challenge / Exercises
1. **Leakage check**: Retrain all models without the `time` column. Does AUC drop significantly?
2. **Threshold tuning**: Adjust the decision threshold from 0.5 to maximise recall for the 'Died' class.
3. **SHAP**: Install `shap` and plot a beeswarm plot for the Gradient Boosting model.
4. **Cross-validation**: Use `StratifiedKFold(n_splits=10)` and report mean ± std AUC for each model.
5. **Feature interaction**: Is there a multiplicative interaction between `ejection_fraction` and `serum_creatinine`?

## 25. Final Summary / Key Takeaways
```
TAKEAWAY 1  Serum creatinine and ejection fraction are the dominant predictors of mortality.
TAKEAWAY 2  Gradient Boosting outperforms simpler models but needs cross-validation to confirm.
TAKEAWAY 3  Be careful with 'time' — including it may inflate performance by leaking outcome info.
TAKEAWAY 4  Class imbalance (~32%) demands AUC and recall as primary metrics, not accuracy.
TAKEAWAY 5  Despite the small dataset, meaningful signal exists for clinical decision support.
```